# Model v9: v4-C + surrogato dell'ottimizzatore + kNN su settimane analoghe
I dati vengono da SHOP (ottimizzatore LP/MIP con water value a fine orizzonte). L'errore di v4-C è di *regime settimanale* (quante unità girano), cioè proprio ciò che l'ottimizzatore decide col bilancio idrico. Due modi nuovi di dare questa informazione al modello:
1. **SURR**: per ogni centrale un dispatch semplificato. Budget d'acqua della settimana tra un minimo (evitare lo sfioro) e un massimo (acqua disponibile = volume + inflow + rilascio atteso delle centrali a monte), con budget "economico" = ore in cui prezzo > water value. Il budget viene versato nelle ore più care (water-filling). Output: unità stimate accese per ora, ore-equivalenti di budget, rilascio a monte in unità equivalenti.
2. **KNN**: settimane storiche più simili (prezzi, volumi, water value, inflow, stagione, giorno della settimana) e media delle loro etichette. Usato sia da solo sia come feature (OOF) di XGBoost.

Restano il consenso tra casi sovrapposti e la calibrazione per unità di v8. FUT (volumi futuri) è stato tolto: usa informazione non disponibile in pianificazione.

In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, pairwise_distances
import joblib
from functools import cached_property
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
import yaml
import xgboost as xgb

In [2]:
# create the classes and funcs to prepare the data
def build_cases(X, Y, T=168):
    '''
    Modify the dataset to have them case indexed
    Input:
    - X: x dataset
    - Y: y dataset
    '''
    
    # splitting between daily and hourly features
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol = [c for c in X.columns if c.endswith('_volume')]
    wv  = [c for c in X.columns if c.endswith('_wv')]

    # getting the position of every case starting hour
    pos = X.index.get_indexer(Y.index)
    assert (pos >= 0).all() and (pos + T < len(X)).all()

    # creating the sliding windows
    win = swv(X[hourly].to_numpy(np.float32), T, axis=0)       
    xh = win[pos].reshape(len(pos), -1)                        
    
    wv_arr = X[wv].to_numpy()
    xs = np.hstack([
        X[vol].to_numpy()[pos],  # initial volume (s)
        wv_arr[pos],             # initial water value (pos)
        wv_arr[pos + T]          # final water value (pos+T)
    ])

    # getting the names for the columns of x
    hourly_cols = [f"{c}_t{h}" for c in hourly for h in range(T)]
    wv_init_cols = [f"{c}_init" for c in wv]
    wv_final_cols = [f"{c}_final" for c in wv]
    
    all_cols = hourly_cols + vol + wv_init_cols + wv_final_cols

    # excluding run_no e total_calculation_time
    y = pd.DataFrame(
        Y.filter(like='result_committed').to_numpy(np.uint8),
        index=Y.index,
        columns=Y.columns[2:]
    )
    
    x = pd.DataFrame(
        np.hstack([xh, xs]).astype(np.float32),
        index=Y.index,
        columns=all_cols
    )
    
    return x, y


class DataLoader:
    # TODO: Change timestamp data from string to datatime object if available in the data source.
    # TODO: move this in the notebooks (or make it simpler/remove it)

    def __init__(self, data_source=None):
        self.data_source = Path(data_source) if data_source else data_dir

    def _csv(self, folder, filename, date_columns=()):
        file_path = self.data_source / folder / filename
        frame = pd.read_csv(file_path)
        for column in date_columns:
            if column in frame.columns:
                frame[column] = pd.to_datetime(frame[column])
        return frame

    def _yaml(self, folder, filename):
        file_path = self.data_source / folder / filename
        with file_path.open(encoding="utf-8") as stream:
            return yaml.safe_load(stream)

    @cached_property
    def day_ahead_price_data(self):
        return self._csv(
            "kernel",
            "Historical_day_ahead_price_2015_2025.csv",
            ("date",),
        )

    @cached_property
    def inflow_data(self):
        return self._csv("kernel", "Historical_inflow_1958_2025.csv", ("date",))

    @cached_property
    def volume_data(self):
        return self._csv("kernel", "Historical_volume_2015_2024.csv", ("date",))

    @cached_property
    def synthetic_water_value_data(self):
        return self._csv(
            "kernel",
            "Synthetic_water_value_2015_2024.csv",
            ("date",),
        )

    @cached_property
    def commitment_descision_data(self):
        return self._csv("kernel", "Unit_commitment_decisions.csv", ("starttime",))

    @cached_property
    def commitment_decision_data(self):
        return self.commitment_descision_data

    @cached_property
    def min_flow_constraint_data(self):
        return self._csv("extended", "Constraint_min_flow.csv", ("date",))

    @cached_property
    def min_volume_constraint_data(self):
        return self._csv("extended", "Constraint_min_volume.csv", ("date",))

    @cached_property
    def topology_data(self): #TODO: CHECK THIS ONE LATER
        return self._yaml("extended", "Tokke_Vinje_topology.yaml")



# Class to create the entire dataset in one go
class Training_set(DataLoader):
    '''
    It imports all the dataset, merging them using the the index (that will be the date)
    '''
 
    def __init__(self, data_source=None):
        super().__init__(data_source)
        self.y = None
        self.x = None
 
    def load_copies(self):
        sources = {
            "day_ahead_price": self.day_ahead_price_data,
            "inflow": self.inflow_data,
            "volume": self.volume_data,
            "wv": self.synthetic_water_value_data,
            "uc": self.commitment_decision_data,
        }
        return sources
 
 
    def create_x(self):
        '''
        Create the complete X dataset
        
        TODO: feature engineering and selection
                + time split for easier computation (test model on smaller data)
        '''
        
        data = self.load_copies()
        
        # setting the date as index and fixing the columns names
        for name in data.keys():
            if 'date' in list(data[name].columns):
                data[name]['date'] = pd.to_datetime(data[name]['date'], utc=True)
                data[name] = data[name].set_index(data[name]['date']).drop(columns='date')
                if 'Day-ahead price (EUR/MWh)' in data[name].columns:
                    data[name].columns = data[name].columns.str.replace(r'\s*\([^)]*\)', '', regex=True).str.replace(r'[\s-]+', '_', regex=True).str.lower()
                else:
                    data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')
                
            else:
                data[name]['starttime'] = pd.to_datetime(data[name]['starttime'], utc=True)
                data[name] = data[name].set_index(data[name]['starttime']).drop(columns='starttime')
                data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')

        # wv and volume have daily data
        #   need to repet thier value hourly
        hourly = data['day_ahead_price'].index
        for name in ('volume', 'wv'):
            data[name] = data[name].reindex(hourly, method='ffill')
        
        # creating the two datset
        #   y stored for future use
        self.y = data['uc'].copy()
        data.pop('uc')
        x = data['day_ahead_price']
        data.pop('day_ahead_price')
        for name in data.keys():
                x = x.join(data[name].add_suffix('_' + name), how='left')
                
        
        self.x = x.copy()
    
    
    def split_small_data(self, begin_train:str ='2015-01-01', end_train: str='2018-12-31', begin_test: str='2019-01-12', end_test: str='2019-12-31'):
        '''
        Return X and Y dataset
        
        Input:
            - beginnig_train (str, dafault=2015-01-01 ): strart date training, format 'yyyy-mm-dd'
            - beginnig_test (str, dafault=2018-12-31 ): strart date testing, format 'yyyy-mm-dd'
            - end_train (str, dafault=2019-01-01 ): end date of training, format 'yyyy-mm-dd'
            - end_test (str, dafault=2019-12-31 ): end date of testing, format 'yyyy-mm-dd'
        '''
        
        self.create_x()
        X, Y = self.x.copy(), self.y.copy()
        x_train, y_train = build_cases(X, Y.loc[begin_train:end_train])
        x_test,  y_test  = build_cases(X, Y.loc[begin_test:end_test])
        return x_train, y_train, x_test, y_test

In [3]:
DATA = './Users/lucaag/data'
ts = Training_set(data_source=DATA)
ts.create_x()
X_all, Y_all = ts.x.sort_index(), ts.y.sort_index()

split = int(len(Y_all) * 0.8)  # stessa split di v4
x_train, y_train = build_cases(X_all, Y_all.iloc[:split])
x_test, y_test = build_cases(X_all, Y_all.iloc[split + 7:])
topo = ts.topology_data['model']
for x in (x_train, x_test): assert (x.index.to_series().diff().dropna() == pd.Timedelta('1D')).all()  # casi consecutivi
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(2337, 3243) (2337, 2352) (578, 3243) (578, 2352)


## Topologia e target (come v4)

In [4]:
T = 168
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}
PN = {k.lower().replace(' ', '_'): g['p_nom'] for k, g in topo['generator'].items()}
TOPO = {
    'vesle_kjela': (['kjelavatn'], 'foersvatn'),
    'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
    'songa': (['songavatn', 'bitdalsvatn'], 'totak'),
    'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
    'haukeli': (['vatjern', 'langeidvatn'], 'vinjevatn'),
    'tokke': (['vinjevatn'], 'bandak'),
    'byrte': (['botnedalsvatn'], 'byrtevatn'),
    'lio': (['byrtevatn'], 'bandak'),
    'hogga': (['bandak'], None),
}
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']

units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in y_train.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
stack3 = lambda y: np.stack([y[ucols(u)].to_numpy() for u in range(len(units))], axis=1)  # (casi, unità, 168)
Y3, Y3te = stack3(y_train), stack3(y_test)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]
auc = lambda Y, P: roc_auc_score(Y.ravel(), P.ravel())  # micro AUC

## Feature v4

In [5]:
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

def shift(a, k):
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

def head(r, v):
    d = RES[r]['vol_head']
    return np.interp(v, d['x'], d['y'])

def static(X, plant):
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    S = [X[[f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]].to_numpy()]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']),
                              X[f'{up[0]}_wv_final'].to_numpy() - wd]))
    return np.hstack(S)

def inflow(X, up):  # inflow orario sommato sui bacini a monte (m3/s)
    return sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
               np.zeros((len(X), T), np.float32))

## Surrogato dell'ottimizzatore
Energia per Mm3 = 2.725·η·H_netta (MWh), quindi la portata massima dell'impianto è `cap = Σp_nom / (2.725·η·H)` in Mm3/h. Per ogni centrale 4 budget d'acqua settimanali: **min** (niente sfioro), **init/final** (ore con prezzo > water value iniziale/finale, limitate tra min e max), **max** (acqua disponibile). Ognuno viene versato nelle ore più care: `unità_t = clip(W/cap − rank_prezzo_t, 0, 1)·n_unità`.

In [6]:
ETA = 0.88

def cap(X, plant):  # portata massima dell'impianto (Mm3/h), stessa testata netta di static()
    up, dn = TOPO[plant]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0
    H = np.maximum(hu - np.maximum(hd, PL[plant]['outlet_line']), 5)
    return sum(PN[units[u]] for u in plant_units[plant]) / (2.725 * ETA * H)

def surr(X, plant, rel=None):  # rel (n,168): rilascio atteso delle centrali a monte (Mm3/h)
    up, c, nu = TOPO[plant][0], cap(X, plant), len(plant_units[plant])
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)
    Win = inflow(X, up).sum(1) * 0.0036 + (rel.sum(1) if rel is not None else 0)  # m3/s * 0.0036 = Mm3/h
    Wmax = np.minimum(V0 + Win, T * c)
    Wmin = np.minimum(np.maximum(V0 + Win - sum(RES[r]['max_vol'] for r in up), 0), Wmax)
    P = X[PRICE].to_numpy(); rank = (-P).argsort(1).argsort(1)
    Wp = [np.clip((P > X[f'{up[0]}_{w}'].to_numpy()[:, None]).sum(1) * c, Wmin, Wmax) for w in ('wv_init', 'wv_final')]
    Ws = [Wmin, *Wp, Wmax]
    S = [np.clip((W / c)[:, None] - rank, 0, 1) * nu for W in Ws]  # unità accese stimate (indice 2 = budget wv_final)
    S += [np.repeat((W / c / T)[:, None], T, 1) for W in Ws]       # frazione di orizzonte a piena capacità
    if rel is not None:
        S += [rel / c[:, None] * nu, np.cumsum(rel, 1) / c[:, None]]  # rilascio a monte in unità equivalenti e in ore cumulate
    return S

def base(X, plant, up_prob=None, rel=None, sur=False):  # matrice (n*168, F); con sur l'ultima colonna è il dispatch stimato
    up, n = TOPO[plant][0], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = inflow(X, up)
    wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7,
         P, shift(P, 1), shift(P, -1), P.argsort(1).argsort(1) / (T - 1),
         D.argsort(2).argsort(2).reshape(n, T) / 23, P - np.repeat(D.mean(2), 24, 1),
         P - P.mean(1, keepdims=True), P - wvf, I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    A = np.column_stack([f.ravel() for f in F] + [np.repeat(static(X, plant), T, 0),
                                                  np.repeat(X.index.dayofyear.to_numpy(), T)])
    if sur:
        S = surr(X, plant, rel)
        A = np.column_stack([A] + [s.ravel() for s in S] + [S[2].ravel()])
    return A.astype(np.float32)

## Training: un modello per centrale, OOF a blocchi contigui, stacking topologico (come v4)

In [7]:
NE, FOLDS, GAP, SEEDS = 200, 5, 7, 1  # SEEDS=3 per la submission finale

def mk(seed):
    return xgb.XGBClassifier(n_estimators=NE, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             tree_method='hist', n_jobs=-1, random_state=seed)

def fp(X, y, Xs):
    if y.min() == y.max():
        return [np.full(len(x), float(np.clip(y[0], .01, .99))) for x in Xs]
    ms = [mk(s).fit(X, y) for s in range(SEEDS)]
    return [np.mean([m.predict_proba(x)[:, 1] for m in ms], 0) for x in Xs]

def folds(n):
    b = np.linspace(0, n, FOLDS + 1, dtype=int)
    for a, c in zip(b[:-1], b[1:]):
        yield np.r_[0:max(a - GAP, 0), min(c + GAP, n):n], np.arange(a, c)

ridx = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
ylong = lambda c, us: np.concatenate([Yflat[u][ridx(c)] for u in us])

def design(B, c, us, K, sur):  # righe (caso, ora) x unità; extra: indice unità, (dispatch stimato - indice), kNN
    r = ridx(c); R = B[r]
    def blk(j, u):
        e = [np.full(len(r), u), np.full(len(r), j)]
        if sur: e.append(R[:, -1] - j)
        if K is not None: e.append(K[c, u].ravel())
        return np.column_stack([R] + e)
    return np.vstack([blk(j, u) for j, u in enumerate(us)])

def fit_group(B, Bt, us, K, Kt, sur):
    n, nt = len(B) // T, len(Bt) // T
    allc = np.arange(n)
    te = fp(design(B, allc, us, K, sur), ylong(allc, us), [design(Bt, np.arange(nt), us, Kt, sur)])[0].reshape(len(us), nt, T)
    oo = np.zeros((len(us), n, T))
    for tr, va in folds(n):
        p = fp(design(B, tr, us, K, sur), ylong(tr, us), [design(B, va, us, K, sur)])[0]
        oo[:, va] = p.reshape(len(us), len(va), T)
    return oo, te

def run(sur=False, knn=False):  # ritorna predizioni test e OOF, entrambe (casi, unità, 168)
    oof, te, rtr, rte = {}, {}, {}, {}
    K, Kt = (K_tr, K_te) if knn else (None, None)
    P = np.zeros((len(x_test), len(units), T), np.float32); O = np.zeros((len(x_train), len(units), T), np.float32)
    for plant in ORDER:
        us, ups = plant_units[plant], UP_PLANTS.get(plant, [])
        m = lambda d: np.mean([d[p] for p in ups], 0) if ups else None
        s = lambda d: sum(d[p] for p in ups) if (ups and sur) else None  # rilascio a monte (Mm3/h)
        oo, t = fit_group(base(x_train, plant, m(oof), s(rtr), sur), base(x_test, plant, m(te), s(rte), sur), us, K, Kt, sur)
        P[:, us, :], O[:, us, :] = t.transpose(1, 0, 2), oo.transpose(1, 0, 2)
        te[plant], oof[plant] = t.mean(0), oo.mean(0)
        rtr[plant], rte[plant] = oof[plant] * cap(x_train, plant)[:, None], te[plant] * cap(x_test, plant)[:, None]
        print(plant, 'ok')
    return P, O

## kNN su settimane analoghe
Distanza su gruppi di feature standardizzati e pesati (`KW`). OOF con gli stessi blocchi di XGBoost. Il numero di vicini `k` si sceglie sull'AUC OOF. Da solo è anche un modello diverso da confrontare con XGBoost.

In [8]:
KW = {'price': 1.5, 'dow': .7, 'vol': 1, 'wv': 1, 'inflow': .7, 'season': .7}

def zfeat(X):
    n, P = len(X), X[PRICE].to_numpy(); D = P.reshape(len(X), 7, 24).mean(2)
    st = lambda s: X[[c for c in X.columns if c.endswith(s)]].to_numpy()
    I = np.column_stack([inflow(X, [r]).sum(1) for r in RES if f'{r}_inflow_t0' in X.columns])
    a = 2 * np.pi * X.index.dayofyear.to_numpy() / 365.25
    return {'price': np.column_stack([D, D.mean(1), D.std(1)]), 'dow': np.eye(7)[X.index.dayofweek.to_numpy()],
            'vol': st('_volume'), 'wv': np.hstack([st('_wv_init'), st('_wv_final')]), 'inflow': I,
            'season': np.column_stack([np.sin(a), np.cos(a)])}

def zmat(Xs):  # standardizza sul train, ogni gruppo pesato e normalizzato per sqrt(dimensione)
    zs, out = [zfeat(x) for x in Xs], [[] for _ in Xs]
    for g, w in KW.items():
        m, s = zs[0][g].mean(0), zs[0][g].std(0) + 1e-6
        for o, z in zip(out, zs): o.append((z[g] - m) / s * w / np.sqrt(z[g].shape[1]))
    return [np.hstack(o) for o in out]

Ztr, Zte = zmat([x_train, x_test])
Yk = Y3.reshape(len(Y3), -1).astype(np.float32)

def knn(Zq, Zr, Yr, k):
    D = pairwise_distances(Zq, Zr); idx = np.argpartition(D, k, 1)[:, :k]
    d = np.take_along_axis(D, idx, 1); w = np.exp(-(d / d.max(1, keepdims=True)) ** 2); w /= w.sum(1, keepdims=True)
    return sum(w[:, [i]] * Yr[idx[:, i]] for i in range(k))

def knn_all(k):
    O = np.zeros_like(Yk)
    for tr, va in folds(len(Ztr)): O[va] = knn(Ztr[va], Ztr[tr], Yk[tr], k)
    return O.reshape(Y3.shape), knn(Zte, Ztr, Yk, k).reshape(len(Zte), *Y3.shape[1:])

KN = {k: knn_all(k) for k in (10, 30, 60)}
for k, (O, P) in KN.items(): print('kNN k =', k, '| AUC oof', round(auc(Y3, O), 4), '| test', round(auc(Y3te, P), 4))
KB = max(KN, key=lambda k: auc(Y3, KN[k][0])); K_tr, K_te = KN[KB]; print('k scelto (su oof):', KB)

kNN k = 10 | AUC oof 0.6866 | test 0.5489
kNN k = 30 | AUC oof 0.6892 | test 0.5838
kNN k = 60 | AUC oof 0.6888 | test 0.6126
k scelto (su oof): 30


## Post-processing: consenso tra casi sovrapposti e calibrazione per unità (come v8)

In [9]:
def consensus(P, b, K=6):
    # P (casi consecutivi, unità, 168). Stesso istante assoluto: caso d+k, ora t-24k. Peso 1 al caso stesso, b a ogni vista
    n = len(P); S = np.zeros_like(P); C = np.zeros((n, 1, T), np.float32)
    for k in range(1, K + 1):
        w = 24 * k
        S[:n-k, :, w:] += P[k:, :, :T-w];  C[:n-k, :, w:] += 1
        S[k:, :, :T-w] += P[:n-k, :, w:];  C[k:, :, :T-w] += 1
    return (P + b * S) / (1 + b * C)

logit = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4))).reshape(-1, 1)

def calib(Poof, Yoof, Pte):  # Platt per unità sulle OOF: monotona, cambia solo l'allineamento tra unità
    out = np.array(Pte, dtype=np.float32)
    for u in range(Pte.shape[1]):
        y = Yoof[:, u].ravel()
        if y.min() == y.max(): continue
        m = LogisticRegression(C=100).fit(logit(Poof[:, u]), y)
        out[:, u] = m.predict_proba(logit(Pte[:, u]))[:, 1].reshape(Pte[:, u].shape)
    return out

## Confronto (v4-C su questa split: 0.9423). Scegliere configurazione e b su `AUC oof`, non su test
Per risparmiare tempo si possono togliere voci da `CFGS` (la riga 'C (v4)' è solo il riferimento).

In [ ]:
CFGS = {'C (v4)': (0, 0), '+SURR': (1, 0), '+KNN': (0, 1), '+SURR+KNN': (1, 1)}
rows, preds = [], {}
for name, (s, k) in CFGS.items():
    P, O = run(bool(s), bool(k)); preds[name] = (P, O)
    for b in (0, .5, 1):
        Pc, Oc = consensus(P, b), consensus(O, b)
        rows += [(name, b, False, auc(Y3, Oc), auc(Y3te, Pc)), (name, b, True, np.nan, auc(Y3te, calib(Oc, Y3, Pc)))]
pd.DataFrame(rows, columns=['modello', 'b', 'calib', 'AUC oof', 'AUC test']).round(4)

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
tokke ok
hogga ok
vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok


,modello,b,calib,AUC oof,AUC test
0,C (v4),0.0,False,0.9612,0.9438
1,C (v4),0.0,True,NaN,0.9381
2,C (v4),0.5,False,0.9277,0.9207
3,C (v4),0.5,True,NaN,0.9147
4,C (v4),1.0,False,0.9151,0.9093
5,C (v4),1.0,True,NaN,0.9044
6,+SURR,0.0,False,0.9716,0.9570
7,+SURR,0.0,True,NaN,0.9531
8,+SURR,0.5,False,0.9391,0.9316
9,+SURR,0.5,True,NaN,0.9292


AUC test per unità (b=0): dove guadagna ogni idea.

In [ ]:
pd.DataFrame({n: [roc_auc_score(Y3te[:, i].ravel(), p[0][:, i].ravel()) for i in range(len(units))]
              for n, p in preds.items()}, index=units).round(4)

,C (v4),+SURR,+KNN,+SURR+KNN
byrte_g1,0.9900,0.9959,0.9877,0.9945
haukeli_g1,0.8876,0.8894,0.8768,0.8828
hogga_g1,0.9498,0.9700,0.9470,0.9675
kjela_g1,0.9811,0.9728,0.9754,0.9755
lio_g1,0.8020,0.8357,0.8062,0.8286
songa_g1,0.9635,0.9379,0.9583,0.9618
tokke_g1,0.9438,0.9770,0.9436,0.9717
tokke_g2,0.9206,0.9650,0.9262,0.9566
tokke_g3,0.9410,0.9740,0.9427,0.9662
tokke_g4,0.9209,0.9646,0.9257,0.9544


## File di submission (dopo aver scelto configurazione e b)

In [ ]:
def to_df(P, index, columns):
    return pd.DataFrame(P.reshape(len(P), -1), index=index, columns=out_cols)[columns]
# P, O = preds['+SURR+KNN']; sub = to_df(calib(consensus(O, b), Y3, consensus(P, b)), y_test.index, y_test.columns)